# KV Cache Multi-Tool Profiler

This notebook drives a matching script benchmark that captures:
- PyTorch Profiler traces with a warmup/active schedule
- Nsight Systems timelines using NVTX ranges for correlation
- Nsight Compute kernel metrics using the same decode-phase ranges

The workload is realistic enough to include:
- KV-cache prefill and decode phases
- parameter sweeps over prompt length and batch size
- NVTX ranges around `prefill`, `decode_loop`, and optionally `decode_step`


In [ ]:
pip install torch transformers tensorboard torch_tb_profiler

In [ ]:
from pathlib import Path
import shutil

ARTIFACTS = Path("./artifacts/kv_cache_profiler")
ARTIFACTS.mkdir(parents=True, exist_ok=True)

PROMPT_LENGTHS = "128,512"
BATCH_SIZES = "1,4"
MAX_NEW_TOKENS = 64
WARMUP_STEPS = 1
ACTIVE_STEPS = 2

print(f"Artifacts dir: {ARTIFACTS.resolve()}")
print("nsys:", shutil.which("nsys"))
print("ncu:", shutil.which("ncu"))


## PyTorch Profiler Run

This writes TensorBoard-compatible traces under `./artifacts/kv_cache_profiler/torch_traces` and a summary CSV under `./artifacts/kv_cache_profiler/`.

In [ ]:
!python ./kv_cache_profiler.py --profile-tool torch --prompt-lengths {PROMPT_LENGTHS} --batch-sizes {BATCH_SIZES} --max-new-tokens {MAX_NEW_TOKENS} --warmup-steps {WARMUP_STEPS} --active-steps {ACTIVE_STEPS}

## TensorBoard

In [ ]:
!nohup tensorboard --logdir ./artifacts/kv_cache_profiler/torch_traces --host 0.0.0.0 --port 6006 > ./artifacts/kv_cache_profiler/tensorboard.log 2>&1 &
!echo "TensorBoard started on port 6006"
!echo "Log file: ./artifacts/kv_cache_profiler/tensorboard.log"

## Nsight Systems

In [ ]:
!nsys profile --trace=cuda,nvtx,osrt --sample=none --output ./artifacts/kv_cache_profiler/nsys_kv_cache_timeline python ./kv_cache_profiler.py --profile-tool none --prompt-lengths {PROMPT_LENGTHS} --batch-sizes {BATCH_SIZES} --max-new-tokens {MAX_NEW_TOKENS} --warmup-steps {WARMUP_STEPS} --active-steps {ACTIVE_STEPS}

## Nsight Compute

In [ ]:
!ncu --set full --nvtx --nvtx-include decode_step --export ./artifacts/kv_cache_profiler/ncu_kv_cache_decode python ./kv_cache_profiler.py --profile-tool none --prompt-lengths {PROMPT_LENGTHS} --batch-sizes {BATCH_SIZES} --max-new-tokens {MAX_NEW_TOKENS} --warmup-steps {WARMUP_STEPS} --active-steps {ACTIVE_STEPS} --mark-decode-steps

## Outputs

Expected artifacts:
- `./artifacts/kv_cache_profiler/kv_cache_profiler_summary_torch.csv`
- `./artifacts/kv_cache_profiler/kv_cache_profiler_summary_none.csv` when running under Nsight tools
- Nsight `.nsys-rep` or `.ncu-rep` files in the same artifact folder